<div
  style="
    background-color: #f0f0f0;
    color:rgb(56, 56, 56);
    padding: 8px;
    display: flex;
    align-items: center;
    gap: 100px;
  "
>
  <img src="./images/brand.svg" style="max-height: 80px;">
  <strong>
    AI Saga: Data Science and Machine Learning</br>
    2.4.1. Polynomial Regression with SGD
  </strong>
</div>

### Background

You will be working with a synthetic dataset generated from a polynomial function with added noise. The dataset includes:

- 1000 data points
- 1 input feature (x)
- 1 target variable (y) generated by a polynomial function
- Random noise to simulate real-world data

### Description

Your task is to explore different approaches to fit this non-linear data using SGDRegressor. You should:

- Start by fitting a linear model and analyze its limitations
- Implement polynomial feature transformation
- Experiment with different learning rates and number of iterations
- Compare the performance of different polynomial degrees
- Visualize the learning process and final predictions

### Deliverables

- A **ipynb** (Jupyter Notebook) file called **polynomial-regression-sgd.ipynb** that contains your code and analysis
- The **notebook** should include visualizations of the different models and their learning curves
- A discussion of the techniques you found most effective for minimizing the error
- Use the provided **polynomial-regression-sgd.ipynb** template as a starting point


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from rich.table import Table
from rich.console import Console

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_squared_error

Visual configuration of Seaborn for better graphics

In [ ]:
sns.set_theme(style="whitegrid", palette="muted")
sns.set_context("notebook", font_scale=1.2)

Rich console instance to display results

In [ ]:
console = Console()

## Reading Dataset

In [ ]:
df = pd.read_csv("./data/synthetic-polynomial-data.txt", sep="\t")
X = df.drop(columns=["y"])
y = df["y"]

console.print(f"Size of dataset: {df.shape}")

## Visualization of data

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(X, y, color="m", marker="x", s=5)
plt.xlabel("x")
plt.ylabel("y")
plt.show()

## 1. Create a linear model and analyze its limitations

Split the data into training and testing sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Standardize features

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

console.print(f"Training set: {X_train.shape}")
console.print(f"Test set: {X_test.shape}")

Method to create and train a SGDRegressor model

In [ ]:
def train_model(learning_rate, alpha, eta0, max_iter, X_trn, y_trn, X_tst, y_tst):

    model = SGDRegressor(
        learning_rate=learning_rate,
        alpha=alpha,
        eta0=eta0,
        max_iter=max_iter,
        random_state=42,
    )

    model.fit(X_trn, y_trn)

    y_tst_pred = model.predict(X_tst)

    mse_tst = mean_squared_error(y_tst, y_tst_pred)

    return y_tst_pred, mse_tst

Train a linear model with SGD Regressor

In [ ]:
lin_y_test_pred, lin_mse_test = train_model(
    "invscaling", 0.0001, 0.01, 1000, X_train, y_train, X_test, y_test
)

console.print(f"MSE train: {lin_mse_test:.2f}")

We plot the test data against the regression line.

In [ ]:
plt.figure(figsize=(12, 5))

plt.scatter(X_test, y_test, color="blue", label="Datos de prueba")
plt.plot(X_test, lin_y_test_pred, color="red", label="Linea de regresion")
plt.xlabel("X")
plt.ylabel("y")
plt.title("Regresion Lineal con SGDRegressor")
plt.legend()
plt.show()

### Observations

When training the model with different configurations using SGDRegressor, the results showed that the prediction error (MSE) varied depending on the parameters used.

This linear model still has limitations, although it manages to capture the general trend between the variables x and y, it does not have a good fit. This is because the actual relationship is curved, while the linear model can only represent it as a straight line. Therefore, it will be necessary to compare its performance with polynomial models, which can better adapt to this type of relationship.

## 2. Implement transformation of polynomial features with different values

In [ ]:
comb_variables = [
    {"learning_rate": "invscaling", "alpha": 0.0001, "eta0": 0.01, "max_iter": 1000},
    {"learning_rate": "constant", "alpha": 0.001, "eta0": 0.1, "max_iter": 2000},
    {"learning_rate": "optimal", "alpha": 0.0005, "eta0": 0.01, "max_iter": 5000},
    {"learning_rate": "adaptive", "alpha": 0.0001, "eta0": 0.5, "max_iter": 3000},
]

polynomial_model_comparisons = []

degrees = [2, 3, 4, 5]

for degree in degrees:

    poly = PolynomialFeatures(degree=degree)
    X_train_poly = poly.fit_transform(X_train)
    X_test_poly = poly.transform(X_test)

    for i, comb in enumerate(comb_variables):
        _, poly_mse_test = train_model(
            comb["learning_rate"],
            comb["alpha"],
            comb["eta0"],
            comb["max_iter"],
            X_train_poly,
            y_train,
            X_test_poly,
            y_test,
        )

        polynomial_model_comparisons.append(
            {"comb": i + 1, "degree": degree, "mse_test": poly_mse_test}
        )

polynomial_model_comparisons.sort(key=lambda x: x["mse_test"])

For the variable combinations, I changed the learning_rate parameter with the options I had seen in resources such as geeksforgeeks and sklearn, such as “optimal,” “constant,” “invscaling,” and “adaptive.” For the alpha and eta0 values, I followed the recommendations given in class by the practitioner (use small values) and seen in resources. For the max_iter value, I tested several ranges and saw that there wasn't much difference with very high values, so I settled on the range of 1000-5000.

This way, I was able to compare the results and see which combinations offered the least error, which allowed me to identify how sensitive the predictions were to changes in the learning rate and other settings.

## 3. Compare the performance of different polynomial degrees

In [ ]:
table = Table(title="Comparison of polynomial degrees")
table.add_column("Combination", style="green")
table.add_column("Degree", style="green")
table.add_column("MSE", style="cyan")

for item in polynomial_model_comparisons:
    table.add_row(
        f"comb_{item["comb"]}", str(item["degree"]), f"{item["mse_test"]:.2f}"
    )

console.print(table)

The best model is the third-degree with the combination of variables 1, as it is the model with the lowest cost. 

## 4. Visualize and compare both models, linear regression versus polynomial regression.

In [ ]:
def render_regression(model_x_test, model_y_test, model_y_pred, model_mse, degree=1):
    plt.scatter(model_x_test, model_y_test, s=30, label="Data")

    sort_idx = np.argsort(model_x_test[:, 0])
    X_test_sorted = model_x_test[sort_idx]
    y_pred_sorted = model_y_pred[sort_idx]

    plt.plot(X_test_sorted, y_pred_sorted, color="r", label="Final Prediction")
    plt.xlabel("X")
    plt.ylabel("y")
    plt.title(
        f"{'Lineal' if degree == 1 else f'Polynomial - Degree {degree}'}, MSE = {model_mse:.2f}"
    )
    plt.legend()

In [ ]:
linear_y_pred, linear_mse = train_model(
    "invscaling", 0.0001, 0.01, 1000, X_train, y_train, X_test, y_test
)

poly = PolynomialFeatures(degree=3)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

poly_y_pred, poly_mse = train_model(
    "invscaling", 0.0001, 0.01, 1000, X_train_poly, y_train, X_test_poly, y_test
)

plt.figure(figsize=(12, 8))

plt.subplot(1, 2, 1)
render_regression(X_test, y_test, linear_y_pred, linear_mse)

plt.subplot(1, 2, 2)
render_regression(X_test, y_test, poly_y_pred, poly_mse, 3)

plt.tight_layout()
plt.show()

## Conclusions

- The third-degree polynomial model fits the data better than the linear model, as it more closely follows the actual shape of the relationship between the variables and fuel consumption.
- During the training process, the choice of the learning rate strategy had a clear impact on performance. Some strategies, like "adaptive", helped the model adjust more flexibly, while others, like "constant", required more careful tuning of parameters such as eta0 to achieve stable results.
- The comparison between the linear and polynomial models showed that the choice of polynomial degree directly influences the model's ability to better represent the data. In this case, degree 3 offered a good balance between simplicity and adjustment capacity.
- The strategies to minimize error must consider both the correct choice of training parameters and the appropriate complexity of the model (linear/polynomial), which allows for more accurate predictions without losing generalization.